# Redshift

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/redshift.ipynb)

Redshift is AWS's analytical data warehouse: columnar, massively parallel SQL with RA3 managed storage separating compute from data. This notebook covers what a data engineer needs: the architecture, distribution styles and sort keys, COPY and UNLOAD, Spectrum, WLM and concurrency, maintenance (VACUUM/ANALYZE), security, and query tuning. Single-option trivia ("what does COPY's IGNOREHEADER do") and per-integration yes/no questions from generic interview lists have been removed; the *Things to remember* sections near the end compress what matters.

The runnable cells model Redshift mechanics locally: pure Python for slice distribution, and [DuckDB](https://duckdb.org) for zone maps, upserts and semi-structured data. Redshift-only DDL (`DISTKEY`, `SORTKEY`, `COPY`, `UNLOAD`) stays in `sql` blocks.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.4"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## What is Amazon Redshift?

The leader node parses SQL, builds and optimizes the plan, compiles it, and coordinates the compute nodes. It holds the catalog and metadata, not user data.

Compute nodes are split into slices. Each slice owns a share of CPU, memory and disk, and a table's rows live on those slices. Adding nodes adds slices, which adds scan bandwidth and memory. Slice counts differ by node type: dc2.large has 2 per node, ra3.xlplus has 4, ra3.4xlarge has 12, and ra3.16xlarge has 48.

MPP means each slice scans only its own blocks and the leader merges the partial results. That is why a scan of one huge fact table still finishes quickly, and why a single oversized input file can bottleneck a load.

RA3 adds Redshift Managed Storage, which separates compute from storage. Local SSD becomes a cache rather than the source of truth, so compute can be resized without copying all the data. AQUA is an RA3 feature that offloads some scans and aggregation to an on-storage cache layer. It is enabled per cluster and has no SQL DDL.

Common mistake: assuming the leader node does the heavy scanning or stores table data. The leader plans and coordinates; the slices do the work and own the rows. Watching leader CPU to explain a slow scan looks in the wrong place. Slice-level metrics and the query plan show where the work actually went.

## How do distribution styles and sort keys affect Redshift performance?

AUTO starts small tables at ALL. A large table becomes AUTO(EVEN), unless Redshift finds a recurring join column and makes it AUTO(KEY). EVEN round-robins rows, which is fine when no key dominates. KEY hashes the distkey so matching values land on the same slice. ALL copies the entire table to every slice and only makes sense for small dimensions.

Sort keys drive zone-map pruning. Compound is the default and follows the leading columns first, which suits date-first range filters. Interleaved gives several columns equal weight, costs more to maintain, and is meant for low-cardinality filters.

```sql
alter table analytics.fact_sales alter diststyle key distkey (product_key);
alter table analytics.dim_product alter diststyle all;

alter table analytics.fact_sales alter sortkey (date_key, store_key);
alter table analytics.fact_sales alter sortkey auto;
alter table analytics.stg_clicks alter sortkey none;
```
Interleaved sort keys can be converted to compound, never the reverse. ALTER and VACUUM cannot run on the same table at the same time. Sort key columns default to RAW encoding.

Common mistake: choosing an interleaved sort key for a high-cardinality column such as an order id or a timestamp. Interleaved is built for low-cardinality filters and needs VACUUM REINDEX, which scans and rebuilds the sort metadata. If the workload filters mainly on one date column, compound is cheaper. Also check that the distkey really is the busiest join key, because a wrong KEY style forces redistribution on every join.

In [ ]:
import zlib
from collections import Counter

SLICES = 8


def slice_for(value) -> int:
    return zlib.crc32(str(value).encode()) % SLICES


orders = [{"order_id": n, "customer_id": n % 4_000, "status": "shipped" if n % 10 else "returned"}
          for n in range(80_000)]


def skew(counts: Counter) -> float:
    """SVV_TABLE_INFO.skew_rows: rows on the fullest slice / rows on the emptiest slice."""
    per_slice = [counts.get(s, 0) for s in range(SLICES)]
    return max(per_slice) / max(min(per_slice), 1)


even = Counter(n % SLICES for n in range(len(orders)))                 # DISTSTYLE EVEN: round robin
by_customer = Counter(slice_for(o["customer_id"]) for o in orders)     # DISTKEY(customer_id)
by_status = Counter(slice_for(o["status"]) for o in orders)            # DISTKEY(status): 2 values
print(f"EVEN skew={skew(even):.2f}  KEY(customer_id) skew={skew(by_customer):.2f}  KEY(status) skew={skew(by_status):.0f}")

# A join on customer_id is slice-local only if both tables distribute on customer_id.
moved_if_even = sum(1 for n, o in enumerate(orders) if n % SLICES != slice_for(o["customer_id"]))
print(f"rows redistributed per join if orders were EVEN: {moved_if_even:,} of {len(orders):,}")
assert skew(even) == 1.0 and skew(by_customer) < 1.2 and skew(by_status) > 100
assert moved_if_even > 0.8 * len(orders)

## How do you load data efficiently into Redshift?

COPY reads S3 in parallel across slices, so the number of files matters as much as the format. Aim for a multiple of the cluster's slice count, roughly one file per slice. A manifest lists the exact set of files and is the safest way to control the load.

Options you will actually use: IGNOREHEADER, DELIMITER, GZIP, ENCRYPTED, TRUNCATECOLUMNS, ACCEPTINVCHARS, EXPLICIT_IDS, REGION, MAXERROR. STATUPDATE ON refreshes table statistics at the end of the load. COMPUPDATE PRESET tells COPY to reuse the table's existing encodings instead of sampling the incoming data.

```sql
copy analytics.stg_orders
from 's3://acme-lake/raw/orders/'
iam_role 'arn:aws:iam::123456789012:role/redshift-staging-role'
csv
ignoreheader 1
delimiter ','
gzip
truncatecolumns
acceptinvchars
maxerror 10;
```

Common mistake: loading one multi-gigabyte CSV file. That is a single slice of work, so the rest of the cluster idles. Another is setting MAXERROR high to get past a bad file. The load succeeds, but bad rows are recorded in STL_LOAD_ERRORS and you learn about them from a report instead of the load. Check STL_LOAD_ERRORS after any load that reports skipped rows.

## How does Amazon Redshift achieve scalability?

Elastic resize adds or removes nodes quickly and keeps the cluster available. Classic resize moves all the data and takes much longer. On RA3, Redshift Managed Storage separates storage from compute, so you resize for CPU and memory without copying the data. Serverless scales by RPUs and has no nodes to choose. Concurrency scaling is a separate axis: it adds transient clusters for read traffic, governed by max_concurrency_scaling_clusters.

```sql
select count(*) as slice_count
from stv_slices;
```
The slice count is the unit that matters. It changes with node type, so a resized cluster plans the same SQL differently. Counting slices after a resize confirms the new parallelism.

Common mistake: resizing compute to fix slow queries that are really a distribution or sort-key problem. More slices spread the same skewed work wider, and the slowest slice still gates the query. Check skew_rows and unsorted in SVV_TABLE_INFO and look at the plan before paying for a resize.

## What is columnar storage, and why is it important in Redshift?

On each slice, a column is stored in 1 MB blocks. Every block keeps min and max metadata, so a range predicate can skip whole blocks that cannot match. Fewer blocks touched means less I/O. Values in one column also share a type and range, so they compress better together than mixed row values would.

Redshift chooses encodings per column. Numeric and date columns default to AZ64, char and varchar to LZO, and sort key, boolean and float columns to RAW. ENCODE AUTO is the table-level default and lets Redshift change encodings as data grows.

```sql
alter table analytics.dim_product alter column category encode zstd;
alter table analytics.fact_sales alter column net_amount encode az64;

analyze compression analytics.fact_sales;
```
ANALYZE COMPRESSION only reports recommended encodings. It does not change any data, so you still apply the suggestions yourself.

Common mistake: expecting a columnar store to be fast at point lookups. A query like `select * from analytics.fact_sales where sale_id = 12345` reads blocks from every column just to assemble one row, which is slower than a row store would be. Columnar wins on wide scans over a few columns, not on fetching single rows.

In [ ]:
import pathlib
import tempfile

import duckdb

con = duckdb.connect()
tmp = pathlib.Path(tempfile.mkdtemp())

# Same rows, written unsorted and sorted by order_date (a SORTKEY). Each row group stands in for a
# 1 MB block with min/max metadata (Redshift's zone map).
source = """
    SELECT i AS order_id, DATE '2026-01-01' + CAST((i * 7919) % 365 AS INTEGER) AS order_date, i % 97 AS amount
    FROM range(200000) t(i)
"""
con.execute(f"COPY ({source}) TO '{tmp}/unsorted.parquet' (FORMAT parquet, ROW_GROUP_SIZE 4096)")
con.execute(f"COPY ({source} ORDER BY order_date) TO '{tmp}/sorted.parquet' (FORMAT parquet, ROW_GROUP_SIZE 4096)")


def blocks_to_read(path: str) -> tuple[int, int]:
    return con.execute(f"""
        SELECT count(*) FILTER (WHERE stats_max_value >= '2026-10-01' AND stats_min_value <= '2026-10-07'),
               count(*)
        FROM parquet_metadata('{path}') WHERE path_in_schema = 'order_date'
    """).fetchone()


unsorted, sorted_ = blocks_to_read(f"{tmp}/unsorted.parquet"), blocks_to_read(f"{tmp}/sorted.parquet")
print(f"one-week filter reads {unsorted[0]}/{unsorted[1]} blocks unsorted, {sorted_[0]}/{sorted_[1]} sorted")
assert unsorted[0] == unsorted[1] and sorted_[0] <= 2

## How can you optimize query performance in Amazon Redshift?

Start with the table-level facts before changing any SQL.

```sql
select "schema", "table", diststyle, sortkey1, size, pct_used,
       unsorted, stats_off, skew_rows, tbl_rows
from svv_table_info
where "schema" = 'analytics'
order by size desc;
```

Read the numbers carefully. size is in 1 MB blocks. stats_off of 0 means current statistics and 100 means out of date. unsorted is the percent of rows not in sort order. skew_rows is the largest slice row count divided by the smallest; a large ratio means one slice does most of the work. ANALYZE is incremental, so refreshing statistics is cheaper than it sounds. VACUUM handles space reclamation and sort order, and VACUUM SORT ONLY or VACUUM DELETE ONLY targets one job at a time.

Common mistake: running a full VACUUM on a schedule regardless of need, or skipping ANALYZE after a large load. VACUUM cannot run inside a transaction block and cannot run at the same time as ALTER on the same table. ANALYZE is the cheap command and fixes most bad plans. Check stats_off and unsorted first, then vacuum only the tables that need it.

## Can you change the sort key or distribution key of an existing table in Redshift?

`ALTER TABLE` and `VACUUM` cannot run concurrently on the same table, so schedule reorganizing changes in their own window instead of alongside a nightly vacuum. Interleaved sort keys are the one-way door: you can alter them to compound, never back. `AUTO` is also available for both diststyle and sortkey. `SVV_ALTER_TABLE_RECOMMENDATIONS` lists tables whose current design the system believes is costing you, which is a reasonable starting point for a review.

```sql
-- Inspect the design and its symptoms before and after a change
select database, schema, table, diststyle, sortkey1, sortkey_num,
       skew_rows, unsorted, stats_off
from svv_table_info
where schema = 'analytics';

-- Undo a hand-picked key and let Redshift decide again
alter table analytics.fact_sales alter diststyle auto;
alter table analytics.fact_sales alter sortkey auto;
```

Common mistake: firing a long `ALTER TABLE ... ALTER DISTSTYLE` while a `VACUUM` or a large `COPY` runs on the same table. The statements queue behind each other and the window blows out. For a big table the rebuild-and-rename pattern stays safer, because you can prepare and load the new table first and spend only a short transaction on the swap.

## How does Redshift Spectrum work?

Pushdown happens at three levels. Partition pruning drops whole S3 prefixes when the predicate matches a partition column, exactly like Hive. Column pruning reads only the referenced columns, but only for columnar formats such as Parquet and ORC; a CSV or JSON table must read the whole object. Row group metadata inside a Parquet file lets Spectrum skip blocks as well, so file size and row group size directly affect scan cost and therefore the bill.

```sql
-- Register one new partition after files land
alter table spectrum_lake.orders
add partition (dt = '2026-10-05')
location 's3://acme-lake/orders/dt=2026-10-05/';

-- Or reconcile every partition in one statement
msck repair table spectrum_lake.orders;
```

Common mistake: storing the S3 data as CSV and expecting Parquet-like behaviour. A text table cannot prune columns, so a query that touches two fields still reads and parses every byte. A second trap is filtering on a function of the partition column, such as `cast(dt as date) = ...`, which stops the planner from pruning partitions at all.

## Can you join tables from Redshift and Redshift Spectrum in a single query?

The optimizer picks a join strategy from the relative sizes. A small local dimension can be broadcast to the Spectrum scan, while a large S3 result may be redistributed or materialized on the cluster. The local table's distkey still governs the local side of the join. Because scanned bytes are billed, pushing the selective filter to the S3 side is the main cost lever, and a query that reads only one or two partitions is far cheaper than one that reads the month.

```sql
-- Filter the S3 side hard and keep the local side narrow
create table analytics.monthly_revenue as
select
  p.category,
  date_trunc('month', s.order_date) as month,
  sum(s.amount) as revenue
from spectrum_lake.orders s
join analytics.dim_product p
  on s.product_id = p.product_id
where s.dt between '2026-09-01' and '2026-09-30'
  and p.is_active = true
group by 1, 2;
```

Common mistake: joining an external table to a local table on mismatched types, such as `varchar` in S3 against `integer` in Redshift. The implicit cast can block pushdown and force a full scan plus conversion on the cluster. Define the external table columns to match, and always confirm the partition column appears directly in the `WHERE` clause.

## How can you monitor the performance of Amazon Redshift?

STL tables such as `STL_QUERY` and `STL_QUERYTEXT` hold one row per query step or segment and are trimmed periodically, so they are good for live debugging but not for a permanent baseline. SVL views are the durable user-facing layer. `SVL_QUERY_SUMMARY` reports one row per step with `avgtime` and `maxtime` in microseconds, plus `rows`, `bytes`, `workmem` and `is_diskbased`. A step with `is_diskbased = 't'` spilled to disk, which usually means the plan needs more memory or a different sort or distribution key. `SVL_QUERY_METRICS_SUMMARY` rolls metrics up per query. `SYS_QUERY_HISTORY` keeps recent completed queries, and `STV_WLM_QUERY_STATE` shows what is running right now.

```sql
-- Find the steps that spilled to disk; these are the real tuning targets
select query, stm, seg, step, avgtime, maxtime, rows, bytes, workmem, is_diskbased
from svl_query_summary
where is_diskbased = 't'
  and query > pg_last_query_id() - 100
order by workmem desc
limit 20;
```

Common mistake: judging cluster health from elapsed time alone. A query can be slow because one step spilled to disk, because a large table is skewed across slices, or because the queue made it wait before it started. Check `SVL_QUERY_SUMMARY` for the spill, `SVV_TABLE_INFO.skew_rows` for the imbalance, and `STL_WLM_QUERY` for queue time. Also remember `SVV_TABLE_INFO` is superuser-only and hides empty tables the caller does not own, so a non-superuser dashboard can silently show nothing.

## What is the purpose of the VACUUM command in Amazon Redshift?

VACUUM cannot run inside a transaction block, so wrapping it in an explicit `BEGIN` and `COMMIT` fails. It also takes a strong lock while it merges: `ALTER TABLE` and `VACUUM` cannot run on the same table at the same time. The default target is 95 percent sorted, and `VACUUM ... TO 75 PERCENT` relaxes that so it finishes sooner on very large tables. Sort key columns default to RAW encoding, so a re-sort really does rewrite them. `VACUUM BOOST` is a variant for large tables that uses the BOOST feature to speed the sort.

```sql
-- Inspect the benefit before paying the I/O cost
select "table", unsorted, vacuum_sort_benefit, tbl_rows, size
from svv_table_info
where schema = 'analytics'
order by vacuum_sort_benefit desc
limit 10;

vacuum boost analytics.fact_sales;
```

Common mistake: running `VACUUM REINDEX` on a table with a compound sort key. REINDEX only applies to interleaved sort keys, so it does nothing useful there. Another common mistake is a blanket nightly VACUUM on every table: it is I/O heavy, and most tables never need it. Pick the few that show a real need through `SVV_TABLE_INFO.unsorted` and `vacuum_sort_benefit`.

## What is the difference between VACUUM and ANALYZE commands in Redshift?

ANALYZE is incremental and samples rows, so it is cheap enough to run after most loads; it refreshes only the statistics that look stale. `SVV_TABLE_INFO.stats_off` shows 0 for current statistics and 100 for fully out-of-date statistics. `ANALYZE COMPRESSION` is a different tool: it only reads the table and reports recommended encodings, it does not collect statistics, and it does not change data. Note the lock difference too: VACUUM cannot run inside a transaction block, while ANALYZE can.

```sql
-- Which tables have stale stats, and what encodings does Redshift suggest?
select schema, "table", stats_off, tbl_rows
from svv_table_info
where stats_off > 10
order by stats_off desc;

analyze analytics.fact_sales;
analyze compression analytics.fact_sales;
```

Common mistake: running `ANALYZE COMPRESSION` and assuming the recommended encodings were applied. It only produces a report; you still apply the change with `ALTER COLUMN ... ENCODE` or by letting COPY run with `COMPUPDATE`. The other common mistake is treating ANALYZE as a substitute for VACUUM. Fresh statistics fix join order and pruning choices, but they do not reclaim space or re-sort unsorted rows.

## How can you secure data in Amazon Redshift?

Row-level security is enforced by a policy that carries a column definition in `WITH` and a predicate in `USING`. The predicate can call `current_user` or read a lookup table instead of relying on a session setting. Column-level grants only restrict access while the role does not also hold table-level SELECT: a table-wide `GRANT SELECT` overrides the column list. Audit logging writes connection, user activity and user logs to S3, separately from CloudTrail, which records the API history.

```sql
-- A mapping table lets one policy serve many regions
create table analytics.user_region (username varchar(64), region varchar(32));

create rls policy region_lookup
  with (region varchar(32))
  using (region in (
    select r.region
    from analytics.user_region r
    where r.username = current_user
  ));

alter table analytics.fact_sales row level security on;
```

Common mistake: granting SELECT on the whole table for convenience and then adding column-level grants expecting them to hide PII. In Redshift the table-level grant wins, so the column list restricts nothing. Audit that no role holds broad table SELECT, then grant only the columns each role actually needs.

## How can you take backups of Amazon Redshift clusters?

```bash
# Automated snapshot retention
aws redshift modify-cluster \
  --cluster-identifier analytics-cluster \
  --automated-snapshot-retention-period 14

# On-demand snapshot before a risky migration
aws redshift create-cluster-snapshot \
  --cluster-identifier analytics-cluster \
  --snapshot-identifier analytics-pre-migration
```

Automated snapshots are created on a schedule and when enough data changes, then kept for the retention period. Manual snapshots are kept until you delete them, which makes them the right tool before a migration or another risky change. The retention that Redshift accepts is 1 to 35 days and defaults to 1. RA3 and Serverless snapshots can be shared across accounts, copied to another region, and restored into a new cluster or namespace.

Common mistake: deleting a cluster and assuming the automated snapshots survive. Deleting a cluster removes its automated snapshots unless you take a final snapshot first; manual snapshots are what remain. Take a manual snapshot before any risky change, and do not run production on the minimum retention.

## What is the purpose of the UNLOAD command in Redshift?

Note that `UNLOAD` is server-side: the data never flows through the client, and it parallelizes automatically. Use `PARALLEL OFF` only when a single ordered file is genuinely required.

UNLOAD writes the result of a query to S3 from the compute nodes, so it never streams through the client. With `PARALLEL ON` it writes one file per slice, which gives the best throughput and lets a downstream engine read the files in parallel. `MAXFILESIZE` caps file size, and each run can emit a manifest that lists the parts. `ALLOWOVERWRITE` lets a rerun replace existing objects in the prefix.
```sql
unload ('select date_key, product_key, sum(amount) as revenue
           from analytics.fact_sales
          where date_key >= 20260101
          group by date_key, product_key')
to 's3://acme-export/product-revenue/'
iam_role 'arn:aws:iam::123456789012:role/redshift-unload-role'
format as csv
header
parallel on
maxfilesize 96 mb
manifest
allowoverwrite;
```

Common mistake: using `PARALLEL OFF` just to get a single file, then reading it back with a parallel engine. The leader node writes that one file serially, so a large export becomes slow, and the single file is a single unit of work for whatever reads it next. Keep parallel output and match the file count to the slice count. Also add `ALLOWOVERWRITE` when a job may rerun, since UNLOAD fails if the prefix already holds objects.

## How can you manage Redshift query concurrency?

Automatic WLM decides queue memory and concurrency on demand. Manual WLM fixes slots and a memory percentage per queue, which is predictable but can strand memory when queues are idle. Query monitoring rules act on a running query: log it, hop it to another queue, or abort it. Concurrency scaling adds transient clusters for read-heavy bursts, and queries that ran on one show `compute_type = 'primary-scale'` in `SYS_QUERY_HISTORY`.
```sql
-- Did queries overflow onto concurrency scaling clusters?
select compute_type, count(*) as queries
from sys_query_history
where start_time >= dateadd(day, -7, getdate())
group by compute_type
order by queries desc;
```

Short query acceleration gives short queries a dedicated express path so they do not sit behind long scans. It is a queue setting, and the cluster predicts which queries are short enough to use it. Query monitoring rules cover runtime, returned rows, and other metrics. Concurrency scaling only helps read queries. A write-heavy workload, such as a large COPY or a long-running UPDATE, waits for its slot instead of scaling out.

Common mistake: looking for a SQL command to change WLM. There is no `ALTER WORKLOAD MANAGEMENT` statement. WLM is cluster configuration, set through the console, the `wlm_json_configuration` parameter group, or the Redshift API. Configure it there and inspect the effect through the WLM system views.

## Can you perform cross-database queries in Amazon Redshift?

Cross-database queries are resolved inside a single cluster. The leader node holds catalog metadata for every local database, so a three-part name is just another namespace to the planner. The rows still sit on the cluster's slices, which means a join across databases redistributes or broadcasts exactly like a join inside one database. Data sharing is the bridge that makes tables from another cluster appear as a database here, which is why a three-part name can reach them; there is no direct path to a foreign cluster's local databases. The feature is available on RA3 and RG provisioned clusters and on Redshift Serverless.

Real SQL, including views and grants, can span databases:
```sql
create view analytics.v_customer_revenue as
select c.customer_id, c.email, sum(s.amount) as revenue
from sales_db.analytics.fact_sales s
join crm_db.public.customers c
  on s.customer_id = c.customer_id
group by c.customer_id, c.email;

grant select on analytics.v_customer_revenue to group reporting;
```

Common mistake: assuming a three-part name reaches any cluster you can connect to. It reaches only databases in the same cluster, plus databases shared in through data sharing, and it is not supported at all on DS2 or DC2. The second trap is privileges: being able to connect is not the same as having SELECT on crm_db.public.customers, so the object rights must be granted in the source database too.

## How can you improve query performance for small tables in Amazon Redshift?

`DISTSTYLE ALL` writes a full copy of the table onto every slice. A join against it becomes a local lookup, so the large table never has to redistribute or broadcast rows for that join. The price is storage multiplied by the slice count and slower writes on every INSERT, COPY, UPDATE and DELETE, because each write touches all copies. `AUTO` distribution starts genuinely small tables at ALL and moves larger tables to AUTO(EVEN), or to AUTO(KEY) on the join column.

Verify the choice from the catalog rather than assuming:
```sql
select "schema", "table", diststyle, size, skew_rows
from svv_table_info
where "schema" = 'analytics'
  and "table" = 'dim_currency';
```
A `size` that is a meaningful fraction of the cluster and a high `skew_rows` ratio are signals that the table is no longer small enough to replicate.

Common mistake: tagging a table ALL because it is small today. A dimension that grows into millions of rows multiplies storage and write cost on every slice, and the benefit evaporates once it no longer fits comfortably in memory. Recheck `diststyle` and `size` as the table grows, and move a grown table back with `alter table analytics.dim_currency alter diststyle auto;`.

## Can you create indexes on tables in Amazon Redshift?

Redshift has no B-tree index. The nearest analogue is the zone map. Each column is stored on a slice in 1 MB blocks, and every block keeps the minimum and maximum value for that column. A range or equality predicate can skip whole blocks whose min and max cannot match, so the physical sort order of a column decides how much I/O a filter avoids. A compound sort key helps mostly through its leading columns; an interleaved sort key spreads the benefit across several columns.

```sql
-- Compound sort key: the default, and the leading columns carry the benefit
create table analytics.fact_events (
  event_date date,
  country    varchar(2),
  event_type varchar(32),
  user_id    bigint
)
compound sortkey (event_date, country, event_type);

-- An interleaved key can be altered to compound; the reverse is not supported
alter table analytics.fact_events alter sortkey compound (event_date);
```

Common mistake: converting a sort key to interleaved to "index" several filter columns. Interleaved keys need `VACUUM REINDEX` to stay effective, cost more to maintain, and only pay off for low-selectivity filters on the interleaved columns. Because ALTER only lets you go from interleaved to compound, not back, the choice is effectively one-way; start compound unless queries genuinely weight several columns equally.

## How can you automate data loading into Amazon Redshift?

The Redshift Data API runs SQL as an asynchronous statement against a provisioned cluster or a Serverless workgroup. It returns an identifier, and the caller polls or waits on an event until the status is FINISHED, FAILED or ABORTED. Credentials are handled by the service, so no JDBC driver, connection pool or VPC path is needed from Lambda, Step Functions or EventBridge Scheduler. Group related statements into one call to cut round trips and keep the whole run idempotent.

The load itself should still be written so a rerun is safe:
```sql
copy analytics.stg_fact_sales
from 's3://acme-lake/curated/fact_sales/dt=2026-10-07/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet;

merge into analytics.fact_sales as t
using analytics.stg_fact_sales as s
  on t.sale_id = s.sale_id
when matched then update set amount = s.amount, order_date = s.order_date
when not matched then insert (sale_id, customer_id, order_date, amount)
values (s.sale_id, s.customer_id, s.order_date, s.amount);

truncate analytics.stg_fact_sales;
```

Common mistake: scheduling COPY with no duplicate guard and no run record. A Lambda retry or a Step Functions catch block can run the same load twice and double the row count. Load into a staging table and MERGE on a natural key, or swap a partition, then write one audit row per run with status, rows loaded and the MAXERROR setting. Alert when MAXERROR is reached instead of letting a partial load pass silently.

## How does Redshift handle data compression?

Compression is chosen per column, not per table, from the data type and the actual value distribution. The documented defaults are AZ64 for numeric and date columns, LZO for char and varchar, and RAW for sort keys, booleans and floats. `ENCODE AUTO` is the table default, so a new column starts unencoded until load or DDL assigns an encoding. `COPY ... COMPUPDATE ON` lets the load choose encodings, and `COMPUPDATE PRESET` keeps whatever is already defined.

`ANALYZE COMPRESSION` is the advisory command; it reports suggested encodings and does not change any data. You can also override one column while leaving the table under ENCODE AUTO:
```sql
-- Report better encodings; no data is modified
analyze compression analytics.fact_sales;

-- Pin one column and let ENCODE AUTO handle the rest
alter table analytics.fact_sales alter column customer_id encode zstd;
```

Common mistake: expecting `COMPUPDATE ON` to fix a bad encoding on a table that already has data. It mainly helps an empty table or newly added columns, so a poor choice made at first load tends to persist. Also avoid TEXT255 or BYTEDICT on a high-cardinality column, and avoid compressing sort key columns away from RAW. Run `ANALYZE COMPRESSION`, then load into a fresh table to re-encode wholesale.

## What is the purpose of the Redshift Query Optimizer?

EXPLAIN prints the plan tree the optimizer chose: scan nodes, join nodes, and an estimated cost. Labels such as DS_BCAST_INNER mean a broadcast, DS_DIST_BOTH means both sides are redistributed, and DS_DIST_NONE means the join ran locally on each slice. Predicates are pushed to the scan so zone maps can skip whole 1 MB blocks. Because the optimizer estimates cardinality from statistics, wrong row counts lead to a bad join order or a broadcast that should have been a local merge join.

```sql
-- Refresh stats only on the columns used for joins and filters
analyze analytics.fact_sales (product_key, date_key);

select "table", tbl_rows, stats_off
from svv_table_info
where "table" = 'fact_sales';
```

Common mistake: trusting the estimated row counts in a plan when `stats_off` is high. A table that has doubled since the last ANALYZE can be estimated at a fraction of its real size, so the optimizer picks a broadcast that floods the network. Re-run ANALYZE after large loads and confirm the plan against the rows each step actually moved.

## Can you perform updates and deletes on tables in Amazon Redshift?

An update or delete is physical work, not an in-place edit. Redshift writes a new version of each changed row and marks the old version deleted. Readers using the same snapshot keep seeing the old rows until the transaction commits, so a query never sees a half-applied change. The dead rows stay inside the 1 MB blocks on the slice, and the table also drifts out of sort order. Look at the table shape before and after cleanup:
```sql
select "table", size, pct_used, unsorted, stats_off, skew_rows, estimated_visible_rows
from svv_table_info
where "table" = 'fact_orders';

vacuum delete only analytics.fact_orders;
vacuum sort only analytics.fact_orders;
```

Transactions use snapshot isolation, so two statements that touch the same rows serialize, and a long `MERGE` can block concurrent writers. The `MERGE` join key must match at most one target row. If the staging table carries duplicate keys, Redshift raises a duplicate key error instead of silently picking a winner. Maintenance also serializes: `VACUUM` and `ALTER TABLE` cannot run concurrently on the same table, so a background vacuum competes with a distribution or sort key change.

Common mistake: Deleting in a loop and expecting the space to come back. Dead rows are reclaimed only by `VACUUM DELETE ONLY`, and each statement adds more delete overhead. When a change touches most of a table, rewrite it instead: `CREATE TABLE AS SELECT` the surviving rows into a new table, then rename it into place.

In [ ]:
con.execute("""
    CREATE TABLE fact_orders AS SELECT * FROM (VALUES
        ('o-1040', 'packed', 10.0), ('o-1041', 'shipped', 25.0), ('o-1042', 'packed', 40.0)
    ) t(order_id, status, amount);
    CREATE TABLE stg_orders AS SELECT * FROM (VALUES
        ('o-1042', 'shipped', 40.0), ('o-1043', 'packed', 15.0)
    ) t(order_id, status, amount);
""")
# The batch-ELT upsert: MERGE from a deduplicated staging table, inside one transaction.
con.execute("BEGIN")
con.execute("""
    MERGE INTO fact_orders t USING stg_orders s ON t.order_id = s.order_id
    WHEN MATCHED THEN UPDATE SET status = s.status, amount = s.amount
    WHEN NOT MATCHED THEN INSERT (order_id, status, amount) VALUES (s.order_id, s.status, s.amount)
""")
con.execute("COMMIT")
rows = con.execute("SELECT order_id, status FROM fact_orders ORDER BY order_id").fetchall()
print(rows)
assert rows[-2:] == [("o-1042", "shipped"), ("o-1043", "packed")] and len(rows) == 4

## Can you create materialized views in Amazon Redshift?

A refresh has two shapes. Incremental refresh reads only the changed base-table rows and patches the view. Full refresh recomputes the whole query. A view whose query shape is not supported for incremental refresh falls back to a full rebuild, which can cost far more than the query it was meant to accelerate. `REFRESH MATERIALIZED VIEW` runs synchronously in your session, while `AUTO REFRESH YES` refreshes in the background and can lag the base tables for a while.

Track which shape each refresh took and how long it ran:
```sql
create materialized view analytics.mv_customer_ltv
diststyle even
auto refresh yes
as
select customer_id, sum(amount) as lifetime_value
from analytics.fact_sales
group by customer_id;

select mv_name, refresh_type, status, start_time, duration
from sys_mv_refresh_history
where mv_name = 'mv_customer_ltv'
order by start_time desc
limit 10;
```

Common mistake: Building a materialized view over staging tables that are truncated and reloaded, or over very volatile base tables. Every load invalidates the view and triggers another refresh, so total maintenance cost can exceed the savings. Point materialized views at stable dimensions and fact tables, and measure refresh duration before rolling one out broadly.

## Can you use Redshift Federated Query to query data in other AWS data sources?

Federated Query is read-only and pull-based. Redshift generates SQL for the remote engine, which runs it and returns matching rows over the network, so only pushdown-able filters and aggregates stay on the source. A join to a local table forces the remote side back to Redshift and redistributes it. That makes federated query good for small dimension lookups and enrichment, and a poor fit for scanning large remote fact tables. The external schema also depends on a Secrets Manager secret and on network reachability from the cluster's VPC.

Use `EXPLAIN` on a federated query to see what was actually pushed to the remote engine:
```sql
explain
select c.customer_id, c.email
from crm.customers c
where c.status = 'active'
  and c.created_at >= date '2026-01-01';

select schemaname, eskind, databasename, esoptions
from svv_external_schemas
where schemaname = 'crm';
```
`SVV_EXTERNAL_SCHEMAS` shows `eskind`, which tells you whether the schema points at a data catalog, a Hive metastore, or a federated relational source.

Common mistake: Joining two large federated tables, or a big local fact table to a federated table, assuming the work executes remotely. The remote result must still cross the network and be redistributed, and a slow remote query holds Redshift resources while it waits. If the remote side is large, stage it locally first.

## What is the purpose of the Redshift EXPLAIN command?

`EXPLAIN` never executes the query. It prints the plan the optimizer chose, in a tree of steps, with an abstract cost and an estimated row count per step. Cost is a relative unit, not milliseconds, so compare plans against each other rather than reading it as a runtime. The useful signals are structural: a broadcast where you expected a co-located join, a nested loop over a large input, or a sequential scan on a table you thought a sort key would prune. Where the estimate comes from stale statistics, the plan is only as good as the last ANALYZE.

```sql
-- Compare the estimate with the runtime counters
select query, segment, step_type, run_time, cpu_time, blocks_read, blocks_to_disk
from stl_query_metrics
where query = pg_last_query_id()
order by run_time desc
limit 20;

-- Step detail: time, memory and whether the step spilled
select query, stm, seg, step, avgtime, maxtime, rows, bytes, workmem, is_diskbased
from svl_query_summary
where query = pg_last_query_id()
order by avgtime desc
limit 10;
```

Common mistake: treating the estimated row count in the plan as fact and skipping the runtime views. A large gap between estimated and actual rows usually means stale statistics or a predicate the optimizer cannot estimate, and the fix is `ANALYZE` on the join and filter columns, not a rewrite. Another mistake is assuming `EXPLAIN` alone reveals skew: cross-slice imbalance shows up in `SVL_QUERY_METRICS.cpu_skew` and `io_skew`, and in `SVV_TABLE_INFO.skew_rows`, not in the plan text.

## How does Redshift handle query optimization for star schema data models?

Three decisions do most of the work on a star schema. First, distribution: put the fact's `DISTKEY` on the column used by the biggest joins, and let small, slowly changing dimensions use `DISTSTYLE ALL` so every slice holds a full copy and joins stay local. Second, sort: a compound sort key with the date column first lets range filters prune 1 MB blocks through zone maps. Third, statistics: the optimizer needs current ANALYZE data to estimate cardinality and pick join order. A star schema stops behaving when a low-cardinality distkey creates slice skew or a large dimension is copied to every slice.

```sql
-- Read the plan, then check the physical shape it was planned from
explain
select s.region, sum(f.net_amount)
from analytics.fact_sales f
join analytics.dim_store s on f.store_key = s.store_key
where f.date_key between 20261001 and 20261031
group by s.region;

select "table", diststyle, sortkey1, skew_rows, stats_off
from svv_table_info
where schema = 'analytics'
order by skew_rows desc;
```

Common mistake: choosing a fact `DISTKEY` on a low-cardinality column such as a status flag or a region, which pushes most rows onto a few slices and shows up as high `skew_rows`. Another is applying `DISTSTYLE ALL` to a dimension that is not small, which multiplies storage and load time by the slice count. A third is ordering a compound sort key by something other than the leading filter column, so the date range scan touches far more blocks than it needs.

## What is the purpose of the Redshift WLM (Workload Management) configuration?

Automatic WLM decides queue memory and concurrency from demand, guided by each queue's priority and its query monitoring rules. Manual WLM fixes slots and memory percentage per queue, which is predictable but wastes capacity when the workload mix changes. Query monitoring rules are the operational control: a rule can log, change a query's priority, or abort it when it exceeds a threshold such as elapsed time or rows returned. Short query acceleration gives fast queries an express path so they skip the main queues. Concurrency scaling, capped by `max_concurrency_scaling_clusters`, adds transient clusters for read-heavy bursts.

```sql
-- Label a session so it lands in the matching queue
set query_group to 'bi';
select s.region, sum(f.net_amount)
from analytics.fact_sales f
join analytics.dim_store s on f.store_key = s.store_key
group by s.region;
reset query_group;

-- Inspect the configured service classes and how a query was handled
select service_class, name, num_query_tasks, query_working_mem
from stv_wlm_service_class_config
order by service_class;

select query_id, service_class_name, compute_type, short_query_accelerated
from sys_query_history
where query_id = pg_last_query_id();
```

Common mistake: looking for an `ALTER WORKLOAD MANAGEMENT` SQL statement. WLM is cluster configuration and is changed through the console, the `wlm_json_configuration` parameter group, or the Redshift API, with the WLM system views used to observe the result. Another mistake is assigning manual memory percentages per queue that do not sum to 100, which makes the cluster reject or reshape the config, and labelling sessions with a query group that no queue matches so they fall through to the default queue.

## Can you use Redshift to query and analyze semi-structured data formats like JSON or Avro?

A SUPER column stores semi-structured values in a compact binary form and is schemaless, so you do not declare nested attributes up front. You navigate it with PartiQL dot and bracket notation and cast scalars with `::varchar` and similar. You can land JSON directly into SUPER with `COPY ... FORMAT AS JSON 'auto'`, or parse a string column at query time with `json_parse`; `can_json_parse` tells you whether a string would parse without raising an error. Because the shape is known only at read time, the optimizer cannot use column statistics on nested attributes, so filter on ordinary columns, especially the sort key, to keep scans narrow.

```sql
-- Land raw JSON events directly into a SUPER column
copy analytics.events_super
from 's3://acme-lake/events-json/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as json 'auto'
compupdate off;

-- Guard against malformed rows when parsing a varchar payload
select event_id,
       json_parse(payload).properties.event_type::varchar as event_type
from analytics.events_raw
where can_json_parse(payload);
```

Common mistake: keeping JSON in a `varchar` column and digging it out with string functions and regexes, which is brittle and hides the structure from tooling. Another is filtering on a nested attribute - `where properties.event_type::varchar = 'click'` - and expecting zone-map pruning; nested access is evaluated per row, so the scan still reads the whole range. PartiQL attribute names are case-sensitive, so `properties.Campaign` is not `properties.campaign`, and mixed-case keys return nulls instead of an error. For files, prefer the matching SerDe for the format rather than casting text.

In [ ]:
# Redshift SUPER + PartiQL navigation, approximated with DuckDB's JSON type and -> / ->> operators.
con.execute("""
    CREATE TABLE events_raw AS SELECT * FROM (VALUES
        ('e1', '{"event_type": "click", "campaign": {"campaign_id": "cmp-7"}}'),
        ('e2', '{"event_type": "view"}'),
        ('e3', 'not json at all')
    ) t(event_id, payload)
""")
rows = con.execute("""
    SELECT event_id,
           payload::JSON ->> '$.event_type'           AS event_type,    -- properties.event_type::varchar
           payload::JSON ->> '$.campaign.campaign_id' AS campaign_id    -- missing paths are NULL
    FROM events_raw
    WHERE json_valid(payload)                                           -- like can_json_parse()
    ORDER BY event_id
""").fetchall()
print(rows)
assert rows == [("e1", "click", "cmp-7"), ("e2", "view", None)]

## How do you diagnose a slow Redshift query?

Slow-query work is a fixed order of checks, cheapest first. Confirm the query really is slow with a query id, read the per-step metrics for one query, check the table layout behind it, and only then rewrite SQL. Most slow queries on a healthy cluster are a layout problem (missing sort key, skewed distribution, stale statistics) or a queue problem (too little memory, too much concurrency), not a bad plan.

Start with the query history. `STL_QUERY` holds one row per query with `elapsed` in microseconds, so the ranking query is short:

```sql
select query, userid, label, starttime, elapsed / 1000000.0 as elapsed_s, aborted
from stl_query
where starttime > dateadd(hour, -24, getdate())
  and query > 0
order by elapsed desc
limit 20;
```

`STL_` views are step and segment level and are trimmed over time. Copy what you need into a permanent audit table if you want a longer window. For query text use `STL_QUERYTEXT` keyed by `query` and `sequence`.

For one query id, get the per-step breakdown. `STL_QUERY_METRICS` reports `run_time` and `cpu_time` in microseconds and `blocks_to_disk` when intermediate results spilled to disk. `SVL_QUERY_SUMMARY` adds `workmem` and the per-step `is_diskbased` flag:

```sql
select query, segment, step_type, run_time, cpu_time, blocks_read, blocks_to_disk
from stl_query_metrics
where query = 918273
order by run_time desc;

select query, stm, seg, step, avgtime, rows, bytes, workmem, is_diskbased
from svl_query_summary
where query = 918273
order by avgtime desc;
```

Then read the table layout with `SVV_TABLE_INFO` (superuser only). `size` is in 1-MB blocks, `skew_rows` is max slice rows divided by min slice rows, `unsorted` is the percent of rows outside sort order, and `stats_off` runs from 0 (current) to 100 (out of date):

```sql
select "table", diststyle, sortkey1, size, tbl_rows,
       skew_rows, unsorted, stats_off, vacuum_sort_benefit
from svv_table_info
where skew_rows > 1.5 or unsorted > 10 or stats_off > 10
order by size desc;
```

In [ ]:
# Flag table-design problems the way SVV_TABLE_INFO reports them.
tables = [
    {"table": "fact_sales", "size_mb": 1_820_000, "skew_rows": 3.4, "unsorted": 28.0, "stats_off": 2.0},
    {"table": "dim_product", "size_mb": 42, "skew_rows": 1.0, "unsorted": 0.0, "stats_off": 0.0},
    {"table": "stg_orders", "size_mb": 610_000, "skew_rows": 1.1, "unsorted": 61.0, "stats_off": 34.0},
]

for t in tables:
    reasons = []
    if t["skew_rows"] > 1.5:
        reasons.append("skewed rows")
    if t["unsorted"] > 10:
        reasons.append("unsorted rows")
    if t["stats_off"] > 10:
        reasons.append("stale stats")
    verdict = ", ".join(reasons) if reasons else "healthy"
    print(f'{t["table"]:<12} {t["size_mb"]:>9,} MB  {verdict}')

flagged = [t["table"] for t in tables if t["skew_rows"] > 1.5]
assert flagged == ["fact_sales"]
assert sum(t["size_mb"] for t in tables) == 2_430_042
assert tables[1]["unsorted"] == 0.0

Check the result cache before you tune anything. A repeated identical query can return from the result cache and look fast without doing any work, and a sudden slowdown can just mean the cache was invalidated by a load. Disable it for a session when you want honest timings:

```sql
set enable_result_cache_for_session to off;
```

Any change to a referenced table invalidates the cached result, and queries that use non-deterministic functions like `random()` are never cached. `SYS_QUERY_HISTORY` reports `result_cache_hit` and `result_cache_query_id`, which points at the query that filled the cache:

```sql
select query_id, result_cache_hit, result_cache_query_id, elapsed_time
from sys_query_history
where result_cache_hit = true
order by start_time desc
limit 20;
```

Common mistake: starting with `EXPLAIN` and rewriting the SQL. `EXPLAIN` shows the plan the optimizer chose, not how long each step took. Read `STL_QUERY_METRICS`, `SVL_QUERY_SUMMARY` and `SVV_TABLE_INFO` first; they usually show the layout or spilling problem in one query.

## How do you choose a distribution style and sort key from query patterns?

Distribution controls data movement; the sort key controls how much data you read. Answer two questions from the real workload, not from the schema diagram. Which column is joined most often and at the largest scale? That suggests `DISTKEY`. Which columns appear in `WHERE` range filters? That suggests `SORTKEY`.

A star-schema table pair shows the pattern. The fact is distributed on its most common join key, and the small dimension is copied to every slice:

```sql
create table analytics.fact_sales (
  sale_id      bigint        encode az64,
  order_ts     timestamp     encode az64 sortkey,
  product_key  integer       encode az64 distkey,
  store_key    integer       encode az64,
  customer_key bigint        encode az64,
  qty          smallint      encode az64,
  amount       decimal(12,2) encode az64
)
diststyle key;

create table analytics.dim_product (
  product_key integer     encode az64 sortkey,
  category    varchar(64) encode lzo,
  brand       varchar(64) encode lzo
)
diststyle all;
```

Use `DISTSTYLE EVEN` when no single join dominates, and `DISTSTYLE AUTO` when you do not yet know the workload. AUTO starts small tables as ALL and can move a large table to EVEN or KEY later.

Worked example. A 2 TB fact on a 12-slice cluster holds about 170 GB per slice. If the nightly query joins the whole fact to `dim_product` on `product_key`, a `KEY(product_key)` distribution lets every slice join its own rows with no network movement. With `EVEN`, the optimizer redistributes the build side on every run. The dimension at 40 MB is cheap to copy, so `ALL` removes its redistribution completely.

The sort key is separate. `SORTKEY (order_ts)` gives each 1-MB column block a tight timestamp range, so `where order_ts >= dateadd(day, -30, getdate())` skips most blocks. Compound sort keys help the leading columns first, so put the range-filter column first. Interleaved sort keys give equal weight to several columns, but they need `VACUUM REINDEX` and only pay off for a few low-cardinality filter columns.

You can change both later without a rebuild. The table stays queryable, though a sort-key change rewrites data and sort-key columns default to `RAW` encoding:

```sql
alter table analytics.fact_sales alter distkey product_key;
alter table analytics.fact_sales alter sortkey (order_ts, store_key);
alter table analytics.fact_sales alter diststyle auto;
```

`ALTER DISTSTYLE`, `ALTER DISTKEY`, `ALTER SORTKEY` and `VACUUM` cannot run on the same table at the same time. Check `SVV_ALTER_TABLE_RECOMMENDATIONS` and `SVL_AUTO_WORKER_ACTION` to see what automatic table optimization has already decided.

Common mistake: distributing on a surrogate primary key that nothing joins on. Every join then redistributes both sides, and rows scatter unevenly because the key is unique. A second mistake is `DISTSTYLE ALL` on a dimension that keeps growing; the copy is multiplied by slice count and loaded on every insert.

## How do you tune Redshift maintenance, COPY and UNLOAD?

Rows become unsorted because `COPY` appends data in file order and deletes leave gaps. Zone maps then lose their value: a block that should hold one narrow range of timestamps instead holds a wide one, so a range filter reads many more blocks. `SVV_TABLE_INFO.unsorted` reports the percentage of rows outside sort order, and `vacuum_sort_benefit` estimates the maximum scan improvement from fixing it.

Worked example. An 800 GB table at 4 percent read for a monthly filter reads about 32 GB when sorted. At 25 percent unsorted, the out-of-order rows are spread across blocks, so the same filter touches closer to 28 percent, or 224 GB. The extra 192 GB is pure maintenance debt.

Run maintenance by need, not by calendar:

```sql
vacuum sort only analytics.fact_sales to 75 percent;
vacuum delete only analytics.fact_sales;
vacuum reindex analytics.dim_product;
analyze analytics.fact_sales;
analyze compression analytics.fact_sales;
```

`VACUUM` reclaims space and re-sorts, `SORT ONLY` only re-sorts, `DELETE ONLY` only reclaims, and `REINDEX` rebuilds an interleaved sort key. `ANALYZE` collects the statistics the optimizer needs for row counts and join order, and `ANALYZE COMPRESSION` only reports suggested encodings. `VACUUM` cannot run inside a transaction block; `ANALYZE` can.

Tune `COPY` for parallelism, not for one big file. Redshift loads one file per slice at a time, so a 900 GB load onto 12 slices with about 3 files per slice means 36 files of roughly 25 GB each. Compressed, columnar files keep the leader node and the network out of the hot path:

```sql
copy analytics.fact_sales
from 's3://acme-lake/curated/fact_sales/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
manifest
format as parquet
compupdate preset
statupdate on
maxerror 0;
```

`MANIFEST` avoids an expensive S3 listing, `COMPUPDATE PRESET` applies the standard encodings without a full scan, and `STATUPDATE ON` refreshes statistics after the load. Use `COMPUPDATE ON` only for a new table whose data shape you have not seen.

`UNLOAD` is the mirror image and writes in parallel across slices:

```sql
unload ('select * from analytics.fact_sales where order_ts >= ''2026-01-01''')
to 's3://acme-lake/exports/fact_sales/'
iam_role 'arn:aws:iam::123456789012:role/redshift-unload-role'
format as parquet
parallel on
maxfilesize 256 mb
manifest
allowoverwrite
partition by (order_ts);
```

`MAXFILESIZE` keeps output files small enough for downstream readers, and `PARTITION BY` writes Hive-style folders that Spectrum and Athena can prune.

Common mistake: loading one multi-terabyte CSV. Only one slice reads it, the rest of the cluster idles, and the leader spends time splitting the stream. Equally bad is thousands of 1 MB files, which spends more time opening objects than reading them.

## How do you tune Redshift WLM, concurrency scaling and Spectrum?

Workload management divides cluster memory and query slots into queues. Automatic WLM allocates memory as queries arrive and picks concurrency on demand; manual WLM fixes `memory_percent` and `max_concurrency` per queue. WLM is cluster configuration, not table DDL: you set it in the console, through the Redshift API, or in the `wlm_json_configuration` cluster parameter. There is no `ALTER WORKLOAD MANAGEMENT` SQL statement; use the JSON config and the WLM system views.

A workable automatic-WLM config separates ETL, BI, and short queries. `short_query_queue` is the short query acceleration (SQA) express lane, which keeps sub-second dashboard queries from waiting behind long scans:

```json
[
  {
    "auto_wlm": true,
    "name": "default",
    "query_group": ["etl"],
    "priority": "high",
    "short_query_queue": false
  },
  {
    "auto_wlm": true,
    "name": "bi",
    "user_group": ["bi_role"],
    "priority": "normal",
    "short_query_queue": true
  }
]
```

Inspect queue state with `STV_WLM_QUERY_STATE` and `STV_WLM_SERVICE_CLASS_CONFIG`. A query that hit a query monitoring rule appears in `STL_WLM_RULE_ACTION`.

Concurrency scaling adds transient clusters when the main cluster's queues are full, so read queries keep starting instead of waiting. It is a cluster-level setting with `max_concurrency_scaling_clusters`, and queries that ran on a scaling cluster show `compute_type = 'primary-scale'` in `SYS_QUERY_HISTORY`. SQA and scaling both help bursty read workloads, but neither fixes a query that scans too much; fix the layout first. AQUA is an RA3 capability that offloads some scans and aggregation to a cache layer in front of Redshift Managed Storage, so it helps large sequential scans on RA3 without any SQL change.

For Spectrum, pruning decides cost because you pay for bytes scanned from S3. Partition on the column you filter most, keep files at a readable size, and store columnar formats so only referenced columns are read:

```sql
create external schema spectrum
from data catalog
database 'analytics_ext'
iam_role 'arn:aws:iam::123456789012:role/redshift-spectrum-role'
create external database if not exists;

create external table spectrum.orders (
  order_id    bigint,
  order_ts    timestamp,
  customer_id bigint,
  amount      decimal(12,2)
)
partitioned by (dt date)
stored as parquet
location 's3://acme-lake/curated/orders/'
table properties ('numRows'='1200000000');
```

`ANALYZE` does not collect external-table statistics. Set `numRows` explicitly so the optimizer does not assume the external table is always the larger side.

Common mistake: treating concurrency scaling as a substitute for query tuning. If every dashboard query scans the full fact table, scaling just buys more clusters doing the same waste. Fix distribution, sort order and statistics, then use scaling and SQA for genuine bursts.

## Things to remember: key numbers and limits

| Topic | Number / fact | Why it matters |
|---|---|---|
| COPY parallelism | one input file per slice in parallel | a single huge file serializes the load onto one slice |
| Distribution | `KEY`, `EVEN`, `ALL`, `AUTO` | decides which joins are local vs broadcast/redistributed |
| Sort keys | compound (leading prefix) vs interleaved; zone maps skip blocks | range filters and joins benefit most |
| Small dimensions | distribute `ALL` (Redshift picks via `AUTO`) | avoids broadcasting a dimension every join |
| No indexes | Redshift has no traditional indexes | performance comes from sort keys, zone maps, and distribution |
| Time travel / recovery | automated snapshots, point-in-time restore | know where snapshots live before deleting a cluster |
| Spectrum billing | per TB scanned from S3, like Athena | Parquet + partition pruning apply the same way |
| Concurrency scaling | transient read clusters absorb bursts | WLM queue time is the symptom to watch |
| Compression | `ANALYZE COMPRESSION` picks encodings per column | loading Parquet or compressed files beats fixing it later |

## Things to remember: semantics to memorize

* **Stage through S3, always.** `COPY` from S3 in parallel is the load path; row-by-row `INSERT` is an anti-pattern for bulk data.
* **`COPY` and `UNLOAD` take an IAM role**, never embedded credentials, and both parallelize across slices — feed them many mid-size files.
* **Zone maps make sort keys work.** Rows in sort order let Redshift skip blocks; an unsorted table reads everything.
* **VACUUM reclaims after large deletes and re-sorts after unsorted loads; ANALYZE refreshes planner statistics** — stale stats produce wrong join orders.
* **The leader node plans and coordinates; it is not the bottleneck to scale first.** Slow queries usually mean skew, missing stats, or a bad distribution, not compute.
* **Redshift is not OLTP.** Single-statement updates and deletes work, but high-frequency row churn belongs in Aurora/RDS with CDC downstream.

## Real world use cases for Redshift

### 1. Retail: store-level sales reporting on RA3

A grocery chain with 800 stores loads 6 TB/month of POS line items and nightly inventory snapshots. The fact table is distributed on `product_key` and sorted by `date_key`; small dimensions use `DISTSTYLE ALL`; cold history older than 24 months lives as Parquet in S3 and is queried through Spectrum.

```sql
create table analytics.fact_pos_lines (
  line_id     bigint,
  date_key    integer,
  store_key   integer,
  product_key integer,
  quantity    integer,
  net_amount  numeric(12,2)
)
diststyle key distkey (product_key)
compound sortkey (date_key, store_key);

-- Store/day rollup served to BI
create materialized view analytics.mv_store_daily
auto refresh yes
as
select date_key, store_key, sum(net_amount) as revenue, sum(quantity) as units
from analytics.fact_pos_lines
group by date_key, store_key;

-- Cold history joined with hot data, no load required
select p.category, sum(s.net_amount) as revenue
from spectrum_lake.pos_history s
join analytics.dim_product p on s.product_key = p.product_key
where s.dt between '2024-01-01' and '2024-12-31'
group by p.category;
```

Why this design: the `product_key` distribution colocates the fact with a `DISTSTYLE ALL` product dimension, so the dominant join has no redistribution; the date-first sort key gives zone-map pruning for the "last N days by store" dashboard; the materialized view keeps the interactive query off the raw fact; and Spectrum keeps the 24-month storage bill on S3 instead of RA3 managed storage.

### 2. Fintech: daily reconciliation with strict freshness and auditability

A payments processor ingests 120 million card transactions per day and must reconcile them against processor settlement files by 06:00, with seven years of queryable history for audits. The load is a single parallel `COPY` from encrypted Parquet, followed by a `MERGE`-based exception step and an `ANALYZE`.

```sql
copy analytics.stg_transactions
from 's3://acme-secure/txn/2026-10-04/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
compupdate preset
maxerror 0;

merge into analytics.fact_transactions t
using (
  select txn_id, account_id, txn_ts, amount, processor_amount, ingested_at,
         row_number() over (partition by txn_id order by ingested_at desc) as rn
  from analytics.stg_transactions
) s
on t.txn_id = s.txn_id and s.rn = 1
when matched then update set t.processor_amount = s.processor_amount
when not matched then insert (txn_id, account_id, txn_ts, amount, processor_amount, ingested_at)
values (s.txn_id, s.account_id, s.txn_ts, s.amount, s.processor_amount, s.ingested_at);

analyze analytics.fact_transactions;

select count(*) as breaks
from analytics.fact_transactions
where amount <> processor_amount and txn_date = current_date - 1;
```

Why this design: `MAXERROR 0` makes a bad settlement file fail the run instead of loading partially; the dedup window plus `MERGE` makes reruns idempotent; the sorting is on the transaction timestamp so audit range queries prune; and audit logging plus cross-region snapshot copy satisfy the retention and DR requirements.

### 3. Marketing: multi-channel attribution and spend reporting

An e-commerce company spends across Google, Meta, and TikTok and loads about 40 GB/day of click and impression data plus 2 GB/day of ad spend. Ad platform files land in S3 as Parquet; attribution runs nightly in Redshift against order data; dashboards query a materialized view.

```sql
create table analytics.fact_touchpoints (
  touch_id    bigint,
  user_id     bigint,
  channel     varchar(32),
  campaign_id varchar(64),
  event_ts    timestamp,
  event_date  date
)
diststyle key distkey (user_id)
compound sortkey (event_date, channel);

create table analytics.fact_spend (
  spend_date  date,
  channel     varchar(32),
  campaign_id varchar(64),
  spend       numeric(12,2)
)
diststyle even sortkey (spend_date, channel);

create materialized view analytics.mv_roas_daily
auto refresh yes
as
select
  t.event_date,
  t.channel,
  count(distinct t.user_id) as reach,
  sum(s.spend) as spend
from analytics.fact_touchpoints t
left join analytics.fact_spend s
  on s.channel = t.channel and s.spend_date = t.event_date
group by t.event_date, t.channel;
```

Why this design: distributing both facts on the channel/date grain keeps the nightly attribution join local; sorting on `event_date` keeps the 30-day lookback scans bounded; and the materialized view gives the marketing team sub-second ROAS without re-running attribution.

### 4. SaaS: product and revenue analytics with governed self-service

A B2B SaaS company stores 3 TB of event and subscription data in Redshift and exposes it to analysts and to embedded customer dashboards. Row-level security isolates tenants, column-level grants hide PII, and WLM separates BI from ETL.

```sql
-- Tenant isolation and column-level least privilege
create role tenant_reader;
grant usage on schema product to role tenant_reader;
grant select (event_date, account_id, event_name, event_count) on product.fact_usage to role tenant_reader;

create rls policy tenant_isolation
  with (account_id varchar(64))
  using (account_id = current_setting('app.account_id'));

attach rls policy tenant_isolation
  on product.fact_usage to role tenant_reader;

-- WLM separation: dashboards get priority, ETL gets a low-priority queue
-- (configured on the cluster; see the WLM question)
```

Why this design: row-level security lets one physical table serve every tenant without duplicating data; WLM priority ensures an ETL backfill cannot degrade customer-facing dashboards; and concurrency scaling absorbs the morning dashboard spike without permanently adding nodes.

### 5. Logistics: fleet telemetry and SLA monitoring

A logistics operator tracks 9,000 vehicles reporting every 15 seconds, about 50 GB/day compressed. Events land in S3 via Firehose, load into a date-sorted fact table, and a nightly job computes on-time delivery SLA by lane. Data older than 90 days is archived with `UNLOAD` to Parquet and dropped from the cluster.

```sql
create table analytics.fact_telemetry (
  vehicle_id  bigint,
  event_ts    timestamp,
  event_date  date,
  lat         numeric(9,6),
  lon         numeric(9,6),
  speed_kph   numeric(6,2),
  fuel_pct    numeric(5,2)
)
diststyle even
compound sortkey (event_date, vehicle_id);

-- Archive before dropping the hot copy
unload ('select * from analytics.fact_telemetry where event_date < current_date - 90')
to 's3://acme-archive/telemetry/'
iam_role 'arn:aws:iam::123456789012:role/redshift-unload-role'
format as parquet
partition by (event_date)
manifest
allowoverwrite;

delete from analytics.fact_telemetry where event_date < current_date - 90;
vacuum delete only analytics.fact_telemetry;
analyze analytics.fact_telemetry;
```

Why this design: a date-first compound sort key makes the "last 24 hours for a lane" range query prune hard; `UNLOAD` to Parquet keeps archive costs on S3 while remaining queryable through Spectrum; and a weekly `VACUUM DELETE ONLY` reclaims the space from the archive delete without a full resort.

## Redshift exercises

These use only the standard library to model distribution, skew, sort order, load parallelism, maintenance debt, queue sizing and Spectrum cost.

### Try it 1. Pick a distribution style from join patterns

**Task.** Write `pick_diststyle(fact_gb, top_join_column, join_share, dim_gb)` for a fact table. Use `KEY(column)` when the dominant join column appears in at least 60 percent of queries, `ALL` when the joined dimension is under 1 GB, and `EVEN` otherwise.

**Hint.** Check the rules in order: dominant join key first, then tiny dimension, then the default.

**Expected result.** `KEY(product_key)`, `ALL`, `EVEN`.

In [ ]:
def pick_diststyle(fact_gb, top_join_column, join_share, dim_gb):
    # KEY colocates the dominant join; ALL copies a tiny dimension; EVEN otherwise.
    if join_share >= 0.6 and top_join_column:
        return f"KEY({top_join_column})"
    if dim_gb < 1:
        return "ALL"
    return "EVEN"


cases = [
    (pick_diststyle(2000, "product_key", 0.82, 0.4), "KEY(product_key)"),
    (pick_diststyle(2000, "product_key", 0.20, 0.4), "ALL"),
    (pick_diststyle(2000, "product_key", 0.20, 80.0), "EVEN"),
]

for got, _ in cases:
    print(got)

for got, expected in cases:
    assert got == expected

### Try it 2. Compute slice skew from row counts

**Task.** Given rows per slice, compute the skew ratio as max rows divided by min rows, the ideal even share, and how many rows sit on the hottest slice above that share.

**Hint.** This is the same `skew_rows` definition as `SVV_TABLE_INFO`. Flag it when the ratio is above 1.5.

**Expected result.** Ratio 1.031, ideal share 9,950,000 rows, excess 150,000 rows.

In [ ]:
rows = [10_000_000, 9_800_000, 10_100_000, 9_900_000]

skew_ratio = max(rows) / min(rows)
ideal_share = sum(rows) / len(rows)
excess = max(rows) - ideal_share

print(f"skew_rows = {skew_ratio:.3f}")
print(f"ideal share = {ideal_share:,.0f} rows")
print(f"excess on hottest slice = {excess:,.0f} rows")

assert round(skew_ratio, 3) == 1.031
assert ideal_share == 9_950_000.0
assert excess == 150_000.0
assert skew_ratio < 1.5

### Try it 3. Choose a sort key from filter columns

**Task.** Write `pick_sortkey(columns)` where `columns` maps a column name to `(filter_count, is_range)`. Order range-filter columns first, then by filter count, so the leading sort-key column is the one that benefits most.

**Hint.** Sort with a key tuple of `(is_range, filter_count)` descending.

**Expected result.** `['order_ts', 'region', 'customer_id']`.

In [ ]:
def pick_sortkey(columns):
    # Range filters benefit most from sort order; then filter frequency.
    ordered = sorted(
        columns.items(),
        key=lambda item: (item[1][1], item[1][0]),
        reverse=True,
    )
    return [name for name, _ in ordered]


columns = {
    "order_ts": (420, True),
    "region": (390, False),
    "customer_id": (120, False),
}

result = pick_sortkey(columns)
print("sortkey =", result)

assert result == ["order_ts", "region", "customer_id"]
assert columns[result[0]][1] is True

### Try it 4. Estimate COPY file count for parallelism

**Task.** A 900 GB load targets a 12-slice cluster with about 3 files per slice. Compute the number of files and the size of each file in MiB. Use 1 GB = 1024 MiB.

**Hint.** `files = slices * files_per_slice`, then `file_mib = total_gb * 1024 / files`.

**Expected result.** 36 files of 25,600 MiB each.

In [ ]:
total_gb = 900
slices = 12
files_per_slice = 3

files = slices * files_per_slice
file_mib = total_gb * 1024 / files

print(f"files = {files}")
print(f"each file = {file_mib:,.0f} MiB")

assert files == 36
assert file_mib == 25_600.0
assert files % slices == 0

### Try it 5. Compute unsorted-rows impact from a table scan

**Task.** An 800 GB table is 25 percent unsorted. A range filter normally reads 4 percent of a sorted table. Model the unsorted rows as spread across all blocks, so the effective read fraction is `sorted_fraction * target + unsorted_fraction`. Compute the bytes read and the extra bytes caused by the unsorted rows.

**Hint.** `sorted_fraction = 1 - unsorted_fraction`. Extra bytes are the difference from the sorted-only read.

**Expected result.** Effective fraction 0.28, 224 GB read, 192 GB extra.

In [ ]:
table_gb = 800
unsorted_fraction = 0.25
target_fraction = 0.04

sorted_fraction = 1 - unsorted_fraction
effective_fraction = sorted_fraction * target_fraction + unsorted_fraction
gb_read = table_gb * effective_fraction
gb_sorted = table_gb * target_fraction

print(f"effective read fraction = {effective_fraction:.2f}")
print(f"bytes read = {gb_read:.0f} GB")
print(f"extra from unsorted rows = {gb_read - gb_sorted:.0f} GB")

assert round(effective_fraction, 2) == 0.28
assert round(gb_read, 6) == 224.0
assert round(gb_read - gb_sorted, 6) == 192.0

### Try it 6. Size a WLM queue from concurrency

**Task.** A BI queue receives 2 queries per second and each runs for 15 seconds. Use Little's law to find the average concurrency, then add a 50 percent safety margin and round up to set `max_concurrency`.

**Hint.** `concurrency = arrival_rate * avg_seconds`; slots are `ceil(concurrency * 1.5)`.

**Expected result.** Average concurrency 30, queue slots 45.

In [ ]:
import math

arrival_per_second = 2.0
avg_seconds = 15.0
safety = 1.5

concurrency = arrival_per_second * avg_seconds
slots = math.ceil(concurrency * safety)

print(f"average concurrency = {concurrency:.0f}")
print(f"queue max_concurrency = {slots}")

assert concurrency == 30.0
assert slots == 45
assert slots > concurrency

### Try it 7. Compare Spectrum and local table scan cost

**Task.** A 100 TB dataset is scanned with Spectrum at an illustrative 5 USD per TB scanned. A local table reads 5 percent of the data thanks to zone maps; the Spectrum table reads 20 percent after partition pruning. Compute both scan costs and the difference.

**Hint.** `cost = table_tb * fraction * usd_per_tb`. The local table has storage cost too, so this compares scan cost only.

**Expected result.** Local 25.0 USD, Spectrum 100.0 USD, Spectrum 75.0 USD more.

In [ ]:
table_tb = 100
local_fraction = 0.05
spectrum_fraction = 0.20
usd_per_tb = 5.0

local_cost = table_tb * local_fraction * usd_per_tb
spectrum_cost = table_tb * spectrum_fraction * usd_per_tb

print(f"local scan cost = {local_cost:.1f} USD")
print(f"Spectrum scan cost = {spectrum_cost:.1f} USD")
print(f"difference = {spectrum_cost - local_cost:.1f} USD")

assert local_cost == 25.0
assert spectrum_cost == 100.0
assert spectrum_cost - local_cost == 75.0